<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_2_MLR_Ames_Part2_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 2
## Forward and Backward Selection with Cross-Validation

**In this notebook we will:**
1.  Load the pre-cleaned, pre-split Ames dataset.
2.  **Shortlist Features:** Use correlation filtering to speed up the selection process (reducing from ~225 to ~40 candidates).
3.  Build a baseline model using forward feature selection and cross-validation.
4.  Diagnose problems with the model: negative coefficients and residual patterns.
5.  Test polynomial features to capture non-linear relationships.
6.  Assess multicollinearity with VIF and correlation analysis.

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download the cleaning module from GitHub
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

# Load and clean the Ames dataset
X_train, X_test, y_train, y_test = load_and_clean_ames()

print(f"X_train shape: {X_train.shape} (Initial high dimensionality)")

### Step 1: Feature Shortlisting (The Speed-Up)

With ~225 features, running Forward Selection (which trains a model for every possible feature combination) can take 20+ minutes. Backward Selection would take even longer.

To speed this up, we'll use a **Correlation Filter**. We calculate the correlation of every feature with our target (`y_train`) and keep only the top 40 most promising candidates. This reduces the search space significantly while keeping the most predictive signals.

In [ ]:
# 1. Calculate absolute correlation of all features with the target
correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)

# 2. Keep the top 40 features as candidates for selection
shortlist = correlations.head(40).index.tolist()

print(f"Shortlisted {len(shortlist)} features for the selection process.")
print("\nTop 10 shortlisted features:")
print(correlations.head(10))

# Create the shortlisted datasets
X_train_short = X_train[shortlist]
X_test_short = X_test[shortlist]

## How Forward and Backward Selection Work

Instead of guessing which features matter, we let the data tell us. These algorithms search through possible feature combinations using cross-validation to evaluate each one.

### Forward Selection (Start Empty, Add Features)
1. Start with no features.
2. Try adding each remaining feature, one at a time.
3. Pick the feature that improves the CV score the most.
4. Repeat until no new feature significantly improves the score.

### Backward Selection (Start Full, Remove Features)
1. Start with all features in the shortlist.
2. Try removing each feature, one at a time.
3. Remove the one whose absence hurts the score the least.
4. Repeat until removing any feature would significantly hurt performance.

In [ ]:
from typing import Dict, Any
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

def perform_feature_selection_and_evaluation(
    X: pd.DataFrame,
    y: pd.Series,
    selection_direction: str = 'forward',
    cv_folds: int = 2
) -> Dict[str, Any]:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SequentialFeatureSelector(
            LinearRegression(),
            n_features_to_select='auto',
            direction=selection_direction,
            cv=cv_folds
        )),
        ('model', LinearRegression())
    ])

    scoring_metrics = ['r2', 'neg_mean_squared_error']
    cv_results = cross_validate(pipeline, X, y, cv=cv_folds, scoring=scoring_metrics)

    cv_r2 = cv_results['test_r2']
    cv_mse = -cv_results['test_neg_mean_squared_error']

    print(f"Cross-validated R-squared: {cv_r2.mean():.4f} \u00b1 {cv_r2.std():.4f}")
    
    pipeline.fit(X, y)
    selected_mask = pipeline.named_steps['selector'].get_support()
    selected_feature_names = X.columns[selected_mask].tolist()

    return {
        'model_pipeline': pipeline,
        'selected_features': selected_feature_names,
        'cv_r2_mean': cv_r2.mean(),
        'final_model': pipeline.named_steps['model']
    }

In [ ]:
def run_modeling_workflow(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    selection_direction: str = 'forward',
    cv_folds: int = 2,
    plot_residuals: bool = True
) -> Dict[str, Any]:
    print(f"--- Running {selection_direction.upper()} Selection ---")
    results = perform_feature_selection_and_evaluation(
        X=X_train, y=y_train, 
        selection_direction=selection_direction, cv_folds=cv_folds
    )

    final_model = results['model_pipeline']
    test_predictions = final_model.predict(X_test)

    print(f"Selected {len(results['selected_features'])} features: {results['selected_features']}")
    print(f"Final Test R-squared: {r2_score(y_test, test_predictions):.4f}")

    if plot_residuals:
        residuals = y_test - test_predictions
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=test_predictions, y=residuals, alpha=0.5)
        plt.axhline(0, color='red', linestyle='--')
        plt.title(f'{selection_direction.capitalize()} Selection Residuals')
        plt.show()

    return results

### Forward Selection
Now that we have a shortlist of 40 features, this will run in seconds!

In [ ]:
forward_results = run_modeling_workflow(
    X_train_short, X_test_short, y_train, y_test,
    selection_direction='forward', cv_folds=2
)

### Backward Selection
Even starting from 40 features, Backward Selection is now very feasible.

In [ ]:
backward_results = run_modeling_workflow(
    X_train_short, X_test_short, y_train, y_test,
    selection_direction='backward', cv_folds=2
)

## Interpreting Results

**Why might Bedroom AbvGr be negative?**
Because `Total_Square_Footage` is already in the model. In two houses with the same total area, the one with *more* bedrooms must have *smaller* rooms. The negative coefficient represents the penalty for cramped layout.

**Residual Plots:**
Because we log-transformed SalePrice in Part 1, we shouldn't see a "funnel shape." The errors should be roughly consistent across the price range.

### Add Polynomials to Model Curves

We'll test if adding squared terms for our top numeric predictors (`Overall Qual`, `Garage Cars`, `Total_Square_Footage`) improves the model.

In [ ]:
from sklearn.preprocessing import StandardScaler

poly_base = ['Overall Qual', 'Garage Cars', 'Total_Square_Footage']
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train[poly_base])
test_scaled = scaler.transform(X_test[poly_base])

X_train_poly = X_train_short.copy()
X_test_poly = X_test_short.copy()

for i, col in enumerate(poly_base):
    X_train_poly[f'{col}_Sqr'] = train_scaled[:, i] ** 2
    X_test_poly[f'{col}_Sqr'] = test_scaled[:, i] ** 2

poly_results = run_modeling_workflow(
    X_train_poly, X_test_poly, y_train, y_test,
    selection_direction='forward', cv_folds=2
)

---

### Assessing Multicollinearity with VIF

Now that our model has selected its favorite features, we check for **Multicollinearity**. If two features overlap too much, their individual coefficients become unreliable.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X: pd.DataFrame):
    X_with_const = X.assign(const=1)
    vif_data = []
    for i, col in enumerate(X.columns):
        vif_value = variance_inflation_factor(X_with_const.values, i)
        vif_data.append({'Feature': col, 'VIF': vif_value})
    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

# Run VIF on the features selected by Forward Selection
final_features = forward_results['selected_features']
# Filter out dummies for cleaner VIF interpretation
numeric_final = [f for f in final_features if X_train[f].nunique() > 2]

print("VIF for selected continuous/ordinal features:")
print(calculate_vif(X_train[numeric_final]))

## Summary

By using **Correlation Filtering** to shortlist candidates, we reduced the runtime of feature selection from 20+ minutes to less than 1 minute. 

1. **Stepwise Selection** pruned the ~40 candidates down to the most essential predictors.
2. **VIF Diagnostics** (run *after* selection) confirmed that the automated process naturally avoided the worst multicollinearity.
3. **Log-Transformation** (from Part 1) kept our residuals stable and our coefficients interpretable as percentage effects.

**Next Step:** In Part 3, we'll see how **Regularization (Lasso/Ridge)** can handle all 225 features at once without needing a shortlist or stepwise selection!